<a href="https://colab.research.google.com/github/gonzaloelejalde/piii-2025/blob/main/clase10/clase10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**4D-PAM5**

Transmisión

In [ ]:
import socket
import numpy as np

# ===== Parámetros =====
HOST = "10.0.0.203"   # IP de la receptora (cambiar por su IP real en LAN)
PORT = 5000
MENSAJE = "Hola Compañera"

# ===== PAM5 =====
PAM5_LEVELS = np.array([-2, -1, 0, 1, 2], dtype=float)

ENC_MAP = {
    (0, 0): -2.0,
    (0, 1): -1.0,
    (1, 0): +1.0,
    (1, 1): +2.0,
}

def string_to_bits(msg):
    return ''.join(format(ord(c), '08b') for c in msg)

def encode_pam5(bits):
    if len(bits) % 2 != 0:
        bits = np.append(bits, 0)
    symbols = []
    for i in range(0, len(bits), 2):
        pair = (int(bits[i]), int(bits[i+1]))
        symbols.append(ENC_MAP[pair])
    return np.array(symbols)

def split_4d(symbols):
    return symbols[0::4], symbols[1::4], symbols[2::4], symbols[3::4]

# ===== Transmisión =====
bits = np.array([int(b) for b in string_to_bits(MENSAJE)])
symbols = encode_pam5(bits)
A, B, C, D = split_4d(symbols)

# Agrupamos todo en una matriz (4 filas = 4 pares UTP)
matrix = np.vstack([A, B, C, D])

# Socket TCP cliente
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((HOST, PORT))
    print("[Tx] Conectado a la receptora")

    # Enviamos la matriz como bytes
    s.sendall(matrix.tobytes())
    print("[Tx] Mensaje transmitido:", MENSAJE)


Recepción

In [ ]:
import socket
import numpy as np
import matplotlib.pyplot as plt

# ===== Parámetros =====
HOST = "0.0.0.0"
PORT = 5000

# ===== PAM5 =====
DEC_MAP = {
    -2.0: (0, 0),
    -1.0: (0, 1),
     1.0: (1, 0),
     2.0: (1, 1),
     0.0: (0, 0),  # opcional: tratar 0 como relleno
}

# ===== Funciones de decodificación =====
def pam5_to_bits(symbols):
    bits = []
    for val in symbols:
        pair = DEC_MAP.get(val, (0, 0))
        bits.extend(pair)
    return bits

def bits_to_string(bits):
    chars = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        if len(byte) == 8:
            chars.append(chr(int(''.join(map(str, byte)), 2)))
    return ''.join(chars)

# ===== Filtro sinc con caída cosenoidal =====
def raised_cosine_sinc(t, T, alpha=0.25):
    sinc = np.sinc(t / T)
    cos = np.cos(np.pi * alpha * t / T)
    denom = 1 - (2 * alpha * t / T)**2
    pulse = sinc * cos / denom
    pulse[np.isnan(pulse)] = 0
    return pulse

def aplicar_filtro_sinc(pam5_data, fs=1000, T=1):
    t = np.linspace(-5*T, 5*T, int(fs*T*10))
    pulse = raised_cosine_sinc(t, T)
    upsampled = np.zeros(len(pam5_data) * int(fs*T))
    upsampled[::int(fs*T)] = pam5_data
    señal = np.convolve(upsampled, pulse, mode='same')
    return señal

def visualizar_senal(señal):
    plt.figure(figsize=(10, 3))
    plt.plot(señal)
    plt.title("📈 Señal PAM5 filtrada (sinc cosenoidal)")
    plt.xlabel("Muestras")
    plt.ylabel("Amplitud")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# ===== Receptor TCP =====
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind((HOST, PORT))
    s.listen()
    print(f"[Rx] Receptor activo en {HOST}:{PORT}. Esperando conexión...")

    conn, addr = s.accept()
    with conn:
        print(f"[Rx] Conectado desde {addr}")
        data = conn.recv(4096)

        # Reconstruir matriz 4xN
        total_vals = len(data) // 8
        symbols = np.frombuffer(data, dtype=np.float64, count=total_vals)
        matrix = symbols.reshape(4, -1)

        print("[Rx] Matriz recibida:")
        print(matrix)

        # Reconstruir secuencia original
        A, B, C, D = matrix
        full_sequence = np.ravel(np.column_stack((A, B, C, D)))

        print("[Rx] Secuencia PAM5 reconstruida:", full_sequence)

        # Aplicar filtro sinc cosenoidal
        señal_filtrada = aplicar_filtro_sinc(full_sequence)
        visualizar_senal(señal_filtrada)

        # Decodificar mensaje
        bits = pam5_to_bits(full_sequence)
        mensaje = bits_to_string(bits)

        print("[Rx] Mensaje decodificado:", mensaje)

**Incluir el uso de un filtro sinc con caída cosenoidal**

Transmisión

In [ ]:
import socket
import numpy as np

# ===== Parámetros =====
HOST = "10.0.0.203"   # IP de la receptora
PORT = 5000
MENSAJE = "Hola Candela"
FS = 1000            # Frecuencia de muestreo
T = 1                # Duración de símbolo (en segundos simulados)
ALPHA = 0.25         # Roll-off del filtro sinc

# ===== PAM5 =====
ENC_MAP = {
    (0, 0): -2.0,
    (0, 1): -1.0,
    (1, 0): +1.0,
    (1, 1): +2.0,
}

def string_to_bits(msg):
    return ''.join(format(ord(c), '08b') for c in msg)

def encode_pam5(bits):
    if len(bits) % 2 != 0:
        bits += '0'
    symbols = []
    for i in range(0, len(bits), 2):
        pair = (int(bits[i]), int(bits[i+1]))
        symbols.append(ENC_MAP[pair])
    return np.array(symbols, dtype=np.float64)

def raised_cosine_sinc(t, T, alpha):
    sinc = np.sinc(t / T)
    cos = np.cos(np.pi * alpha * t / T)
    denom = 1 - (2 * alpha * t / T)**2
    pulse = sinc * cos / denom
    pulse[np.isnan(pulse)] = 0
    return pulse

def generar_senal(symbols, fs, T, alpha):
    t = np.linspace(-5*T, 5*T, int(fs*T*10))
    pulse = raised_cosine_sinc(t, T, alpha)
    upsampled = np.zeros(len(symbols) * int(fs*T))
    upsampled[::int(fs*T)] = symbols
    señal = np.convolve(upsampled, pulse, mode='same')
    return señal

# ===== Transmisión =====
bits = string_to_bits(MENSAJE)
symbols = encode_pam5(bits)
señal = generar_senal(symbols, FS, T, ALPHA)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((HOST, PORT))
    print("[Tx] Conectado a la receptora")
    s.sendall(señal.astype(np.float64).tobytes())
    print("[Tx] Señal sinc transmitida con mensaje:", MENSAJE)

Recepción

In [ ]:
import socket
import numpy as np
import matplotlib.pyplot as plt

# ===== Parámetros =====
HOST = "0.0.0.0"
PORT = 5000
FS = 1000
T = 1
ALPHA = 0.25
PAM5_LEVELS = np.array([-2, -1, 0, 1, 2], dtype=np.float64)

# ===== PAM5 Decodificación =====
DEC_MAP = {
    -2.0: (0, 0),
    -1.0: (0, 1),
     1.0: (1, 0),
     2.0: (1, 1),
     0.0: (0, 0),
}

def pam5_to_bits(symbols):
    bits = []
    for val in symbols:
        pair = DEC_MAP.get(val, (0, 0))
        bits.extend(pair)
    return bits

def bits_to_string(bits):
    chars = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        if len(byte) == 8:
            chars.append(chr(int(''.join(map(str, byte)), 2)))
    return ''.join(chars)

def visualizar_senal(señal):
    plt.figure(figsize=(10, 3))
    plt.plot(señal)
    plt.title("📈 Señal sinc recibida")
    plt.xlabel("Muestras")
    plt.ylabel("Amplitud")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def decision_por_umbral(señal, niveles):
    return [niveles[np.argmin(np.abs(niveles - s))] for s in señal]

def detectar_offset(señal, muestras_por_simbolo):
    # Buscar el primer pico significativo (no solo el más alto)
    ventana = muestras_por_simbolo * 4
    for i in range(muestras_por_simbolo):
        muestras = señal[i::muestras_por_simbolo][:10]
        if np.std(muestras) > 0.5:  # heurística: variación suficiente
            return i
    return 0

# ===== Receptor TCP =====
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind((HOST, PORT))
    s.listen()
    print(f"[Rx] Receptor activo en {HOST}:{PORT}. Esperando conexión...")

    conn, addr = s.accept()
    with conn:
        print(f"[Rx] Conectado desde {addr}")
        data = b""
        while True:
            packet = conn.recv(4096)
            if not packet:
                break
            data += packet

        señal_recibida = np.frombuffer(data, dtype=np.float64)
        print(f"[Rx] Señal recibida con {len(señal_recibida)} muestras")
        visualizar_senal(señal_recibida)

        muestras_por_simbolo = int(FS * T)
        offset = detectar_offset(señal_recibida, muestras_por_simbolo)
        print(f"[Rx] Offset de sincronización detectado: {offset}")

        simbolos_recuperados = señal_recibida[offset::muestras_por_simbolo]
        simbolos_decodificados = decision_por_umbral(simbolos_recuperados, PAM5_LEVELS)

        # Asegurar múltiplo de 4 símbolos
        while len(simbolos_decodificados) % 4 != 0:
            simbolos_decodificados.pop()

        bits = pam5_to_bits(simbolos_decodificados)
        print("[Rx] Bits reconstruidos:", bits)

        mensaje = bits_to_string(bits)
        print("[Rx] Mensaje decodificado:", mensaje)